<a href="https://colab.research.google.com/github/corrielynnyuill-debug/Assignment14-CLY/blob/main/Assignment14_CorrieLynnYuill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load dataset
from sklearn.datasets import fetch_openml

adult = fetch_openml(name='adult', version=2, as_frame=True)
df = adult.frame
print(df.head())

# Basic cleaning
df = df.replace('?', np.nan)
df = df.dropna()

# define target and sensitive attribute
X = df.drop('class', axis=1)
y = df['class']

sensitive_attribute = df['sex']

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sens_train = sensitive_attribute.loc[X_train.index]
sens_test = sensitive_attribute.loc[X_test.index]

# Build logistic regression pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train the model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation metrics
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))










   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female             0             0   

   hours-per-week native-country  class  
0       

In [6]:
# Fairness Analysis
!pip install fairlearn

from fairlearn.metrics import MetricFrame, selection_rate_group_metric
from fairlearn.metrics import accuracy_score, recall_score
import matplotlib.pyplot as plt

# Prepare predictions and sensitve attribute
y_pred = model.predict(X_test)

sens_test = sensitive_attribute.loc[X_test.index]
sens_test.value_counts()

# Define metric functions
def true_positive_rate(y_true, y_pred):
  return recall_score(y_true, y_pred, pos_label='>50K')

def false_positive_rate(y_true, y_pred):
  return recall_score(y_true, y_pred, pos_label='<50K')

# Build MetricFrame
metrics = {
    'selection_rate': selection_rate_group_metric,
    'Accuracy': accuracy_score,
    'True Positive Rate': true_positive_rate,
    'False Positive Rate': false_positive_rate
}

metric_frame = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sens_test
)

print('Overall Metrics:\n', metric_frame.overall)
print('Group Metrics:\n', metric_frame.by_group)

# Plots
plt.figure(figsize=(12, 6))

# Selection Rate
plt.subplot(1, 3, 1)
metric_frame.by_group('selection_rate'.plot(kind='bar'))
plt.title('Selection Rate by Sex')
plt.ylabel('Selection Rate')
plt.xticks(rotation=0)


# True Positive Rate
plt.subplot(1, 3, 2)
metric_frame.by_group('True Positive Rate'.plot(kind='bar'))
plt.title('True Positive Rate by Sex')
plt.ylabel('True Positive Rate')
plt.xticks(rotation=0)

# False Positive Rate
plt.subplot(1, 3, 3)
metric_frame.by_group('False Positive Rate'.plot(kind='bar'))
plt.title('False Positive Rate by Sex')
plt.ylabel('False Positive Rate')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()
#

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 808.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 23.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


ImportError: cannot import name 'selection_rate_group_metric' from 'fairlearn.metrics' (/usr/local/lib/python3.12/dist-packages/fairlearn/metrics/__init__.py)